In [6]:
import os
import subprocess

URL_REPOSITORIO = "https://github.com/railan-freire03/PROJETO_ZOONOSES.git"
PASTA_PROJETO = "/content/PROJETO_ZOONOSES"

# Verifica se já existe uma pasta de tentativa anterior
if os.path.exists(PASTA_PROJETO):
    if os.path.isdir(os.path.join(PASTA_PROJETO, ".git")):
        print("Repositório já está clonado.")
    elif not os.listdir(PASTA_PROJETO):
        os.rmdir(PASTA_PROJETO)
    else:
        raise RuntimeError(
            "A pasta já existe e contém arquivos, mas não é um repositório Git. "
            "Não será apagada automaticamente."
        )

# Clona somente se ainda não estiver clonado
if not os.path.isdir(os.path.join(PASTA_PROJETO, ".git")):
    resultado = subprocess.run(
        ["git", "clone", URL_REPOSITORIO, PASTA_PROJETO],
        capture_output=True,
        text=True
    )

    if resultado.returncode != 0:
        print("ERRO AO CLONAR:")
        print(resultado.stderr)
    else:
        print("Repositório clonado com sucesso!")

if os.path.isdir(PASTA_PROJETO):
    print("\nArquivos encontrados:")
    for item in os.listdir(PASTA_PROJETO):
        print("-", item)

Repositório clonado com sucesso!

Arquivos encontrados:
- notebooks
- codigos
- README.md
- documentacao
- .git


In [11]:
import os

CAMINHO = "/content/PROJETO_ZOONOSES/banco_zoonoses_v2.db"

print("Banco disponível:", os.path.isfile(CAMINHO))
print("Pasta disponível:", os.path.isdir("/content/PROJETO_ZOONOSES"))

Banco disponível: False
Pasta disponível: True


In [9]:

# ================================================================
# PROJETO DE ZOONOSES
# GERADOR DE DADOS BRUTOS — SQLITE
# VERSÃO 2.3 — COMPATÍVEL COM BANCO 2.2 E TRATAMENTO 1.4
#
# - Geração inicial e incremental.
# - Preservação dos lotes e registros anteriores.
# - IDs técnicos sequenciais ANM00000001.
# - Recuperação de IDs por _meta.id_animal_origem.
# - Problemas controlados por conjunto.
# - Duplicidades exatas e parciais.
# - Relacionamentos entre os cinco conjuntos.
# - Vacinas compatíveis com cada espécie.
# - Gravação transacional de lote, brutos e auditoria.
# ================================================================


# 1. IMPORTAÇÕES E CONFIGURAÇÕES

import os
import re
import json
import math
import random
import sqlite3

from copy import deepcopy
from datetime import date, datetime, timedelta


PASTA_PROJETO = "/content/PROJETO_ZOONOSES"

CAMINHO_BANCO = os.path.join(
    PASTA_PROJETO,
    "banco_zoonoses_v2.db"
)

QUANTIDADE_ANIMAIS = 10000

TAXA_PROBLEMAS = 0.12
TAXA_DUPLICADOS = 0.02

# INICIAL: somente para o primeiro lote de um banco vazio.
# ADICIONAR: acrescenta um novo lote sem apagar os anteriores.
MODO_GERACAO = "INICIAL"

VERSAO_GERADOR = "2.3"


# 2. VALIDAÇÕES DAS CONFIGURAÇÕES

if not os.path.isdir(PASTA_PROJETO):
    raise FileNotFoundError(
        "Pasta do projeto não encontrada. "
        "Execute primeiro o notebook do banco de dados."
    )
import os

print("VERIFICAÇÃO DOS CAMINHOS")
print("=" * 50)

print("PASTA_PROJETO:", PASTA_PROJETO)
print("CAMINHO_BANCO:", CAMINHO_BANCO)

print("\nPasta do projeto existe?")
print(os.path.isdir(PASTA_PROJETO))

print("\nBanco no caminho esperado pelo construtor:")
CAMINHO_CORRETO = "/content/PROJETO_ZOONOSES/banco_zoonoses_v2.db"
print(os.path.isfile(CAMINHO_CORRETO))

print("\nBanco no caminho esperado pelo gerador:")
print(os.path.isfile(CAMINHO_BANCO))

print("\nArquivos encontrados na pasta do projeto:")

if os.path.isdir("/content/PROJETO_ZOONOSES"):
    for raiz, pastas, arquivos in os.walk("/content/PROJETO_ZOONOSES"):
        for arquivo in arquivos:
            if arquivo.endswith((".db", ".sqlite", ".sqlite3")):
                print(os.path.join(raiz, arquivo))
if not os.path.isfile(CAMINHO_BANCO):
    raise FileNotFoundError(
        "Banco SQLite não encontrado. "
        "Execute primeiro o construtor do banco."
    )

if (
    not isinstance(QUANTIDADE_ANIMAIS, int)
    or isinstance(QUANTIDADE_ANIMAIS, bool)
    or QUANTIDADE_ANIMAIS <= 0
):
    raise ValueError(
        "QUANTIDADE_ANIMAIS deve ser um inteiro maior que zero."
    )

if not 0 <= TAXA_PROBLEMAS <= 1:
    raise ValueError(
        "TAXA_PROBLEMAS deve estar entre 0 e 1."
    )

if not 0 <= TAXA_DUPLICADOS <= 1:
    raise ValueError(
        "TAXA_DUPLICADOS deve estar entre 0 e 1."
    )

if MODO_GERACAO not in {"INICIAL", "ADICIONAR"}:
    raise ValueError(
        "MODO_GERACAO deve ser INICIAL ou ADICIONAR."
    )


# 3. CONEXÃO COM O SQLITE

conexao = sqlite3.connect(
    CAMINHO_BANCO,
    timeout=60
)

conexao.row_factory = sqlite3.Row

conexao.execute("PRAGMA foreign_keys = ON")
conexao.execute("PRAGMA busy_timeout = 60000")

cursor = conexao.cursor()


# 4. FUNÇÕES DE ESTRUTURA DO BANCO

def obter_tabelas():

    return {
        linha["name"]
        for linha in cursor.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
        """).fetchall()
    }


def obter_colunas(tabela):

    return {
        linha["name"]
        for linha in cursor.execute(
            f'PRAGMA table_info("{tabela}")'
        ).fetchall()
    }


def validar_estrutura():

    tabelas_obrigatorias = {
        "lotes",
        "registros_brutos",
        "animais",
        "auditoria",
        "controle_sequencias",
        "mapeamento_animais"
    }

    existentes = obter_tabelas()

    faltantes = tabelas_obrigatorias - existentes

    if faltantes:
        raise RuntimeError(
            "Tabelas obrigatórias ausentes no banco: "
            + ", ".join(sorted(faltantes))
        )

    colunas_obrigatorias = {

        "lotes": {
            "id_lote",
            "tipo_lote",
            "quantidade_informada",
            "quantidade_gerada",
            "data_criacao",
            "descricao",
            "status"
        },

        "registros_brutos": {
            "id_registro",
            "id_lote",
            "conjunto",
            "dados_json",
            "data_recebimento"
        },

        "auditoria": {
            "entidade",
            "id_registro",
            "operacao",
            "descricao",
            "data_operacao"
        },

        "controle_sequencias": {
            "nome",
            "proximo_valor"
        },

        "mapeamento_animais": {
            "id_animal_origem",
            "id_animal_final",
            "id_lote",
            "data_mapeamento"
        },

        "animais": {
            "id_animal",
            "origem"
        }
    }

    problemas = []

    for tabela, esperadas in colunas_obrigatorias.items():

        existentes_colunas = obter_colunas(tabela)

        ausentes = esperadas - existentes_colunas

        if ausentes:
            problemas.append(
                f"{tabela}: {sorted(ausentes)}"
            )

    if problemas:
        raise RuntimeError(
            "Estrutura incompatível com o Banco 2.2:\n"
            + "\n".join(problemas)
        )

    # Confere também se as tabelas de destino do tratamento
    # estão disponíveis no banco.

    tabelas_tratamento = {
        "animais",
        "vacinacoes",
        "diagnosticos",
        "maus_tratos",
        "obitos_eutanasias"
    }

    faltantes_tratamento = (
        tabelas_tratamento - existentes
    )

    if faltantes_tratamento:
        raise RuntimeError(
            "Tabelas necessárias ao Tratamento 1.4 ausentes: "
            + ", ".join(sorted(faltantes_tratamento))
        )


# 5. REFERÊNCIAS DOS DADOS SINTÉTICOS

MUNICIPIOS = [
    "Feijó",
    "Cruzeiro do Sul",
    "Sena Madureira",
    "Rio Branco",
    "Tarauacá"
]

ESPECIES = [
    "Cachorro",
    "Gato"
]

NOMES_CACHORROS = [
    "Rex", "Thor", "Bob", "Max", "Toby", "Bidu",
    "Pingo", "Fred", "Zeus", "Billy", "Scooby",
    "Bolinha", "Tito", "Apolo", "Bento", "Nino",
    "Marley", "Simba", "Lucky", "Pipoca",
    "Caramelo", "Duque", "Costelinha", "Toddy",
    "Paçoca", "Pudim", "Fubá", "Boris"
]

NOMES_GATOS = [
    "Mimi", "Luna", "Nina", "Mel", "Mingau",
    "Frajola", "Tom", "Garfield", "Chico",
    "Pérola", "Amora", "Salem", "Fumaça",
    "Pipoca", "Biscoito", "Tigrinho", "Nala",
    "Simba", "Mittens", "Félix", "Pudim",
    "Bolinha", "Manchinha", "Cacau", "Floquinho"
]

RACAS_CACHORROS = [
    "Sem raça definida",
    "Pinscher",
    "Poodle",
    "Labrador",
    "Golden Retriever",
    "Pastor Alemão",
    "Shih-tzu",
    "Dachshund",
    "Pit Bull",
    "Beagle",
    "Border Collie",
    "Rottweiler"
]

RACAS_GATOS = [
    "Sem raça definida",
    "Siamês",
    "Persa",
    "Maine Coon",
    "Angorá",
    "Bengal",
    "Ragdoll"
]

MOTIVOS_ACOLHIMENTO = [
    "Maus-tratos",
    "Abandono",
    "Acidente",
    "Doença",
    "Recolhimento preventivo",
    "Entrega voluntária",
    "Animal perdido",
    "Resgate de filhotes",
    "Outros"
]

ORIGENS_ACOLHIMENTO = [
    "Resgate pela equipe",
    "Denúncia da população",
    "Entrega pelo tutor",
    "Órgão público",
    "Clínica veterinária",
    "Recolhimento nas ruas"
]

VACINAS_POR_ESPECIE = {
    "Cachorro": [
        "Antirrábica",
        "V8",
        "V10"
    ],

    "Gato": [
        "Antirrábica",
        "Polivalente felina",
        "Leucemia felina"
    ]
}

DIAGNOSTICOS = [
    "Leishmaniose",
    "Esporotricose",
    "Erliquiose",
    "Cinomose",
    "Parvovirose",
    "Dermatite",
    "Sarna",
    "Verminose",
    "Infecção respiratória",
    "Gastroenterite"
]

TIPOS_MAUS_TRATOS = [
    "Abandono",
    "Agressão física",
    "Negligência",
    "Falta de alimentação",
    "Falta de água",
    "Confinamento inadequado",
    "Ausência de atendimento veterinário"
]

MOTIVOS_OBITO = [
    "Doença",
    "Acidente",
    "Complicações clínicas",
    "Causas naturais"
]

JUSTIFICATIVAS_EUTANASIA = [
    "Sofrimento irreversível",
    "Doença em estágio terminal",
    "Condição clínica sem possibilidade de recuperação"
]


# 6. FUNÇÕES AUXILIARES

def agora():

    return datetime.now().isoformat(
        timespec="seconds"
    )


def data_aleatoria(inicio="2022-01-01", fim=None):

    if fim is None:
        fim = date.today().isoformat()

    data_inicio = datetime.strptime(
        inicio,
        "%Y-%m-%d"
    ).date()

    data_fim = datetime.strptime(
        fim,
        "%Y-%m-%d"
    ).date()

    if data_inicio > data_fim:
        raise ValueError(
            "Data inicial posterior à data final."
        )

    diferenca = (data_fim - data_inicio).days

    return (
        data_inicio
        + timedelta(days=random.randint(0, diferenca))
    ).isoformat()


def data_futura(dias=90):

    return (
        date.today()
        + timedelta(days=random.randint(1, dias))
    ).isoformat()


def peso_aleatorio(especie, raca):

    if especie == "Gato":
        return round(random.uniform(2.0, 8.0), 2)

    if raca == "Pinscher":
        return round(random.uniform(2.0, 6.0), 2)

    if raca == "Dachshund":
        return round(random.uniform(4.0, 12.0), 2)

    if raca in {
        "Labrador",
        "Golden Retriever",
        "Pastor Alemão",
        "Rottweiler"
    }:
        return round(random.uniform(18.0, 38.0), 2)

    if raca == "Pit Bull":
        return round(random.uniform(15.0, 35.0), 2)

    return round(random.uniform(3.0, 25.0), 2)


def criar_meta(id_origem):

    return {
        "id_animal_origem": id_origem,
        "problemas_introduzidos": []
    }


def registrar_problema(registro, problema):

    meta = registro.setdefault("_meta", {})

    problemas = meta.setdefault(
        "problemas_introduzidos",
        []
    )

    if problema not in problemas:
        problemas.append(problema)


def extrair_numero_id(identificador):

    if not identificador:
        return 0

    resultado = re.fullmatch(
        r"ANM(\d+)",
        str(identificador).strip()
    )

    if resultado:
        return int(resultado.group(1))

    return 0


# 7. CONTROLE SEGURO DA SEQUÊNCIA

def recuperar_proximo_id():

    cursor.execute("""
        INSERT OR IGNORE INTO controle_sequencias
        (nome, proximo_valor)
        VALUES ('animais', 1)
    """)

    linha = cursor.execute("""
        SELECT proximo_valor
        FROM controle_sequencias
        WHERE nome = 'animais'
    """).fetchone()

    proximo = int(linha["proximo_valor"])

    maior_existente = 0

    # Animais já tratados.
    for linha in cursor.execute("""
        SELECT id_animal
        FROM animais
    """).fetchall():

        maior_existente = max(
            maior_existente,
            extrair_numero_id(linha["id_animal"])
        )

    # IDs dos animais nos registros brutos.
    for linha in cursor.execute("""
        SELECT dados_json
        FROM registros_brutos
        WHERE conjunto = 'animais'
    """).fetchall():

        try:
            registro = json.loads(linha["dados_json"])

            maior_existente = max(
                maior_existente,
                extrair_numero_id(
                    registro.get("id_animal")
                )
            )

            meta = registro.get("_meta") or {}

            maior_existente = max(
                maior_existente,
                extrair_numero_id(
                    meta.get("id_animal_origem")
                )
            )

        except (json.JSONDecodeError, TypeError):
            continue

    # Mapeamentos antigos também são considerados.
    for linha in cursor.execute("""
        SELECT id_animal_origem
        FROM mapeamento_animais
    """).fetchall():

        maior_existente = max(
            maior_existente,
            extrair_numero_id(
                linha["id_animal_origem"]
            )
        )

    proximo = max(
        proximo,
        maior_existente + 1
    )

    return proximo


# 8. GERAÇÃO DOS ANIMAIS

def gerar_animal(numero):

    especie = random.choice(ESPECIES)

    if especie == "Cachorro":
        nome = random.choice(NOMES_CACHORROS)
        raca = random.choice(RACAS_CACHORROS)
    else:
        nome = random.choice(NOMES_GATOS)
        raca = random.choice(RACAS_GATOS)

    municipio = random.choice(MUNICIPIOS)

    identificador = f"ANM{numero:08d}"

    return {
        "id_animal": identificador,
        "nome": nome,
        "especie": especie,
        "raca": raca,
        "sexo": random.choice(["Macho", "Fêmea"]),
        "municipio": municipio,
        "peso_kg": peso_aleatorio(especie, raca),
        "status": "Ativo",
        "vacinas_faltantes": "",
        "situacao_vacinacao": random.choice([
            "Em dia",
            "Pendente",
            "Parcial"
        ]),
        "imagem": (
            f"{municipio.replace(' ', '_')}"
            f"_animal_{numero:08d}.jpg"
        ),
        "data_acolhimento": data_aleatoria(),
        "motivo_acolhimento": random.choice(
            MOTIVOS_ACOLHIMENTO
        ),
        "origem_acolhimento": random.choice(
            ORIGENS_ACOLHIMENTO
        ),
        "_meta": criar_meta(identificador)
    }


# 9. GERAÇÃO DAS VACINAÇÕES

def gerar_vacinacoes(animal):

    registros = []

    quantidade = random.choices(
        [0, 1, 2, 3],
        weights=[15, 30, 35, 20]
    )[0]

    for _ in range(quantidade):

        data_aplicacao = data_aleatoria(
            inicio=animal["data_acolhimento"]
        )

        proxima_dose = (
            datetime.strptime(
                data_aplicacao,
                "%Y-%m-%d"
            ).date()
            + timedelta(
                days=random.choice([30, 60, 90, 365])
            )
        ).isoformat()

        registros.append({
            "id_animal": animal["id_animal"],
            "nome_vacina": random.choice(
                VACINAS_POR_ESPECIE[animal["especie"]]
            ),
            "data_aplicacao": data_aplicacao,
            "proxima_dose": proxima_dose,
            "situacao": random.choice([
                "Aplicada",
                "Pendente",
                "Aguardando reforço"
            ]),
            "observacoes": "Registro sintético de vacinação.",
            "_meta": criar_meta(
                animal["_meta"]["id_animal_origem"]
            )
        })

    return registros


# 10. GERAÇÃO DOS DIAGNÓSTICOS

def gerar_diagnosticos(animal):

    registros = []

    if random.random() < 0.12:

        registros.append({
            "id_animal": animal["id_animal"],
            "data_diagnostico": data_aleatoria(
                inicio=animal["data_acolhimento"]
            ),
            "diagnostico": random.choice(DIAGNOSTICOS),
            "descricao": (
                "Diagnóstico sintético para fins acadêmicos."
            ),
            "situacao": random.choice([
                "Em tratamento",
                "Em acompanhamento",
                "Recuperado"
            ]),
            "origem": "Gerado artificialmente",
            "_meta": criar_meta(
                animal["_meta"]["id_animal_origem"]
            )
        })

    return registros


# 11. GERAÇÃO DE MAUS-TRATOS

def gerar_maus_tratos(animal):

    registros = []

    gerar = (
        animal["motivo_acolhimento"] == "Maus-tratos"
        or random.random() < 0.025
    )

    if gerar:

        registros.append({
            "id_animal": animal["id_animal"],
            "municipio": animal["municipio"],
            "data_ocorrencia": data_aleatoria(
                inicio=animal["data_acolhimento"]
            ),
            "tipo_ocorrencia": random.choice(
                TIPOS_MAUS_TRATOS
            ),
            "descricao": (
                "Ocorrência sintética para análise."
            ),
            "situacao": random.choice([
                "Em investigação",
                "Em acompanhamento",
                "Concluída"
            ]),
            "origem": "Gerado artificialmente",
            "_meta": criar_meta(
                animal["_meta"]["id_animal_origem"]
            )
        })

    return registros


# 12. GERAÇÃO DE ÓBITOS E EUTANÁSIA

def gerar_eventos(animal):

    registros = []

    if random.random() < 0.015:

        tipo = random.choices(
            ["Óbito", "Eutanásia"],
            weights=[85, 15]
        )[0]

        if tipo == "Óbito":
            motivo = random.choice(MOTIVOS_OBITO)
            observacoes = "Evento sintético de óbito."
        else:
            motivo = random.choice(
                JUSTIFICATIVAS_EUTANASIA
            )
            observacoes = "Evento sintético de eutanásia."

        data_evento = data_aleatoria(
            inicio=animal["data_acolhimento"]
        )

        animal["status"] = "Inativo"

        registros.append({
            "id_animal": animal["id_animal"],
            "tipo_evento": tipo,
            "data_evento": data_evento,
            "municipio": animal["municipio"],
            "motivo": motivo,
            "observacoes": observacoes,
            "_meta": criar_meta(
                animal["_meta"]["id_animal_origem"]
            )
        })

    return registros


# 13. PROBLEMAS CONTROLADOS

PROBLEMAS_POR_CONJUNTO = {

    "animais": [
        "nome_nulo",
        "raca_vazia",
        "peso_negativo",
        "peso_texto",
        "municipio_invalido",
        "sexo_invalido",
        "status_invalido",
        "situacao_vacinacao_invalida",
        "data_acolhimento_futura",
        "motivo_acolhimento_invalido",
        "origem_acolhimento_invalida",
        "id_animal_ausente",
        "id_animal_malformado"
    ],

    "vacinacoes": [
        "vacina_nula",
        "data_aplicacao_futura",
        "proxima_dose_anterior",
        "situacao_invalida",
        "id_animal_ausente"
    ],

    "diagnosticos": [
        "diagnostico_vazio",
        "data_diagnostico_futura",
        "situacao_invalida",
        "id_animal_ausente"
    ],

    "maus_tratos": [
        "municipio_invalido",
        "tipo_ocorrencia_vazio",
        "data_ocorrencia_futura",
        "situacao_invalida",
        "id_animal_ausente"
    ],

    "obitos_eutanasias": [
        "motivo_ausente",
        "data_evento_anterior_ao_acolhimento",
        "municipio_invalido",
        "tipo_evento_invalido",
        "id_animal_ausente"
    ]
}


def modificar_registro(registro, tipo, problema):

    if tipo == "animais":

        alteracoes = {
            "nome_nulo": ("nome", None),
            "raca_vazia": ("raca", ""),
            "peso_negativo": (
                "peso_kg",
                -round(random.uniform(1, 20), 2)
            ),
            "peso_texto": (
                "peso_kg",
                "peso desconhecido"
            ),
            "municipio_invalido": (
                "municipio",
                "Município inexistente"
            ),
            "sexo_invalido": (
                "sexo",
                "Não identificado??"
            ),
            "status_invalido": (
                "status",
                "Desconhecido"
            ),
            "situacao_vacinacao_invalida": (
                "situacao_vacinacao",
                "Vacinação incorreta"
            ),
            "data_acolhimento_futura": (
                "data_acolhimento",
                data_futura()
            ),
            "motivo_acolhimento_invalido": (
                "motivo_acolhimento",
                "Motivo não cadastrado"
            ),
            "origem_acolhimento_invalida": (
                "origem_acolhimento",
                "Origem desconhecida"
            ),
            "id_animal_ausente": (
                "id_animal",
                None
            ),
            "id_animal_malformado": (
                "id_animal",
                "ID-ANIMAL-ERRO"
            )
        }

    elif tipo == "vacinacoes":

        alteracoes = {
            "vacina_nula": ("nome_vacina", None),
            "data_aplicacao_futura": (
                "data_aplicacao",
                data_futura()
            ),
            "proxima_dose_anterior": (
                "proxima_dose",
                "2020-01-01"
            ),
            "situacao_invalida": (
                "situacao",
                "Situação desconhecida"
            ),
            "id_animal_ausente": (
                "id_animal",
                None
            )
        }

    elif tipo == "diagnosticos":

        alteracoes = {
            "diagnostico_vazio": ("diagnostico", ""),
            "data_diagnostico_futura": (
                "data_diagnostico",
                data_futura()
            ),
            "situacao_invalida": (
                "situacao",
                "Situação incorreta"
            ),
            "id_animal_ausente": (
                "id_animal",
                None
            )
        }

    elif tipo == "maus_tratos":

        alteracoes = {
            "municipio_invalido": (
                "municipio",
                "Local desconhecido"
            ),
            "tipo_ocorrencia_vazio": (
                "tipo_ocorrencia",
                ""
            ),
            "data_ocorrencia_futura": (
                "data_ocorrencia",
                data_futura()
            ),
            "situacao_invalida": (
                "situacao",
                "Situação não cadastrada"
            ),
            "id_animal_ausente": (
                "id_animal",
                None
            )
        }

    else:

        alteracoes = {
            "motivo_ausente": ("motivo", None),
            "data_evento_anterior_ao_acolhimento": (
                "data_evento",
                "2020-01-01"
            ),
            "municipio_invalido": (
                "municipio",
                "Município inexistente"
            ),
            "tipo_evento_invalido": (
                "tipo_evento",
                "Evento desconhecido"
            ),
            "id_animal_ausente": (
                "id_animal",
                None
            )
        }

    campo, valor = alteracoes[problema]

    registro[campo] = valor

    registrar_problema(
        registro,
        problema
    )


def aplicar_problemas(registros, tipo):

    if not registros:
        return 0

    quantidade = math.ceil(
        len(registros) * TAXA_PROBLEMAS
    )

    selecionados = random.sample(
        range(len(registros)),
        min(quantidade, len(registros))
    )

    for indice in selecionados:

        problema = random.choice(
            PROBLEMAS_POR_CONJUNTO[tipo]
        )

        modificar_registro(
            registros[indice],
            tipo,
            problema
        )

    return len(selecionados)


# 14. GERAÇÃO DAS DUPLICIDADES

def gerar_duplicidades(animais_originais):

    quantidade = math.ceil(
        len(animais_originais) * TAXA_DUPLICADOS
    )

    exatas = []
    parciais = []

    selecionados = random.sample(
        animais_originais,
        min(quantidade, len(animais_originais))
    )

    for animal in selecionados:

        copia = deepcopy(animal)

        if random.random() < 0.5:

            # A cópia permanece idêntica ao registro original.
            # Isso permite que o tratamento reconheça a duplicidade
            # por comparação do conteúdo.

            exatas.append(copia)

        else:

            campo = random.choice([
                "nome",
                "raca",
                "peso_kg",
                "municipio",
                "sexo"
            ])

            if campo == "nome":
                copia["nome"] = "Nome divergente"

            elif campo == "raca":
                copia["raca"] = "Raça divergente"

            elif campo == "peso_kg":
                copia["peso_kg"] = round(
                    random.uniform(1, 50),
                    2
                )

            elif campo == "municipio":
                copia["municipio"] = random.choice([
                    municipio
                    for municipio in MUNICIPIOS
                    if municipio != animal["municipio"]
                ])

            elif campo == "sexo":
                copia["sexo"] = (
                    "Fêmea"
                    if animal["sexo"] == "Macho"
                    else "Macho"
                )

            registrar_problema(
                copia,
                "duplicidade_parcial"
            )

            parciais.append(copia)

    return exatas, parciais


# 15. EXECUÇÃO PRINCIPAL

try:

    validar_estrutura()

    # O bloqueio é adquirido antes da leitura da sequência.
    # Isso evita que duas execuções utilizem o mesmo próximo ID.

    cursor.execute("BEGIN IMMEDIATE")

    # ------------------------------------------------------------
    # 15.1. VERIFICAÇÃO DO MODO
    # ------------------------------------------------------------

    quantidade_brutos_anteriores = cursor.execute("""
        SELECT COUNT(*)
        FROM registros_brutos
    """).fetchone()[0]

    if (
        MODO_GERACAO == "INICIAL"
        and quantidade_brutos_anteriores > 0
    ):
        raise RuntimeError(
            "Este banco já possui registros brutos. "
            "Para acrescentar outro lote, altere "
            "MODO_GERACAO para 'ADICIONAR'. "
            "Para começar do zero, utilize outro arquivo de banco."
        )

    # ------------------------------------------------------------
    # 15.2. RECUPERAÇÃO DO PRÓXIMO ID
    # ------------------------------------------------------------

    proximo_numero = recuperar_proximo_id()

    print("=" * 65)
    print("GERADOR DE DADOS BRUTOS — VERSÃO 2.3")
    print("=" * 65)

    print("\nIniciando geração dos dados sintéticos...")
    print(f"Modo: {MODO_GERACAO}")
    print(f"Primeiro ID: ANM{proximo_numero:08d}")

    # ------------------------------------------------------------
    # 15.3. GERAÇÃO DOS CONJUNTOS
    # ------------------------------------------------------------

    animais_originais = []
    vacinacoes = []
    diagnosticos = []
    maus_tratos = []
    obitos_eutanasias = []

    for i in range(QUANTIDADE_ANIMAIS):

        numero = proximo_numero + i

        animal = gerar_animal(numero)

        animais_originais.append(animal)

        vacinacoes.extend(
            gerar_vacinacoes(animal)
        )

        diagnosticos.extend(
            gerar_diagnosticos(animal)
        )

        maus_tratos.extend(
            gerar_maus_tratos(animal)
        )

        obitos_eutanasias.extend(
            gerar_eventos(animal)
        )

        if (i + 1) % 1000 == 0:
            print(
                f"{i + 1} animais gerados..."
            )

    # ------------------------------------------------------------
    # 15.4. INCONSISTÊNCIA DE STATUS APÓS EVENTO TERMINAL
    # ------------------------------------------------------------

    animais_com_evento = {
        evento["_meta"]["id_animal_origem"]
        for evento in obitos_eutanasias
    }

    animais_por_origem = {
        animal["_meta"]["id_animal_origem"]: animal
        for animal in animais_originais
    }

    candidatos = [
        animais_por_origem[id_origem]
        for id_origem in animais_com_evento
        if id_origem in animais_por_origem
    ]

    quantidade_status_incorreto = math.ceil(
        len(candidatos) * 0.10
    )

    for animal in random.sample(
        candidatos,
        min(
            quantidade_status_incorreto,
            len(candidatos)
        )
    ):

        animal["status"] = "Ativo"

        registrar_problema(
            animal,
            "animal_ativo_apos_obito_ou_eutanasia"
        )

    # ------------------------------------------------------------
    # 15.5. PROBLEMAS CONTROLADOS
    # ------------------------------------------------------------

    relatorio_problemas = {}

    relatorio_problemas["animais"] = aplicar_problemas(
        animais_originais,
        "animais"
    )

    relatorio_problemas["vacinacoes"] = aplicar_problemas(
        vacinacoes,
        "vacinacoes"
    )

    relatorio_problemas["diagnosticos"] = aplicar_problemas(
        diagnosticos,
        "diagnosticos"
    )

    relatorio_problemas["maus_tratos"] = aplicar_problemas(
        maus_tratos,
        "maus_tratos"
    )

    relatorio_problemas["obitos_eutanasias"] = aplicar_problemas(
        obitos_eutanasias,
        "obitos_eutanasias"
    )

    # ------------------------------------------------------------
    # 15.6. DUPLICIDADES
    # ------------------------------------------------------------

    duplicados_exatos, duplicados_parciais = (
        gerar_duplicidades(animais_originais)
    )

    animais = (
        animais_originais
        + duplicados_exatos
        + duplicados_parciais
    )

    # ------------------------------------------------------------
    # 15.7. ORGANIZAÇÃO DOS CINCO CONJUNTOS
    # ------------------------------------------------------------

    conjuntos = {
        "animais": animais,
        "vacinacoes": vacinacoes,
        "diagnosticos": diagnosticos,
        "maus_tratos": maus_tratos,
        "obitos_eutanasias": obitos_eutanasias
    }

    # ------------------------------------------------------------
    # 15.8. SERIALIZAÇÃO E VALIDAÇÃO DOS JSON
    # ------------------------------------------------------------

    registros_para_inserir = []

    for nome_conjunto, registros in conjuntos.items():

        for registro in registros:

            dados_json = json.dumps(
                registro,
                ensure_ascii=False,
                allow_nan=False
            )

            # Confirma que o JSON pode ser interpretado.
            json.loads(dados_json)

            registros_para_inserir.append((
                nome_conjunto,
                dados_json
            ))

    # ------------------------------------------------------------
    # 15.9. ATUALIZAÇÃO DA SEQUÊNCIA
    # ------------------------------------------------------------

    proxima_sequencia = (
        proximo_numero + QUANTIDADE_ANIMAIS
    )

    cursor.execute("""
        UPDATE controle_sequencias
        SET proximo_valor = ?
        WHERE nome = 'animais'
    """, (
        proxima_sequencia,
    ))

    # ------------------------------------------------------------
    # 15.10. CRIAÇÃO DO LOTE
    # ------------------------------------------------------------

    data_criacao = agora()

    dados_lote = {
        "tipo_lote": "Geração sintética",
        "quantidade_informada": QUANTIDADE_ANIMAIS,
        "quantidade_gerada": len(animais),
        "data_criacao": data_criacao,
        "descricao": (
            f"Gerador {VERSAO_GERADOR}. "
            "Dados sintéticos de zoonoses com cinco conjuntos, "
            "problemas controlados e duplicidades."
        ),
        "status": "Concluído"
    }

    campos_lote = list(dados_lote.keys())

    cursor.execute("""
        INSERT INTO lotes (
            tipo_lote,
            quantidade_informada,
            quantidade_gerada,
            data_criacao,
            descricao,
            status
        )
        VALUES (?, ?, ?, ?, ?, ?)
    """, [
        dados_lote[campo]
        for campo in campos_lote
    ])

    id_lote = cursor.lastrowid

    # ------------------------------------------------------------
    # 15.11. INSERÇÃO DOS REGISTROS BRUTOS
    # ------------------------------------------------------------

    data_recebimento = agora()

    cursor.executemany("""
        INSERT INTO registros_brutos (
            id_lote,
            conjunto,
            dados_json,
            data_recebimento
        )
        VALUES (?, ?, ?, ?)
    """, [
        (
            id_lote,
            conjunto,
            dados_json,
            data_recebimento
        )
        for conjunto, dados_json in registros_para_inserir
    ])

    quantidade_inserida = cursor.rowcount

    if quantidade_inserida != len(registros_para_inserir):
        raise RuntimeError(
            "A quantidade inserida não corresponde "
            "à quantidade preparada."
        )

    # ------------------------------------------------------------
    # 15.12. AUDITORIA
    # ------------------------------------------------------------

    descricao_auditoria = (
        f"Lote {id_lote}. "
        f"Gerador {VERSAO_GERADOR}. "
        f"Animais originais: {len(animais_originais)}. "
        f"Duplicidades exatas: {len(duplicados_exatos)}. "
        f"Duplicidades parciais: {len(duplicados_parciais)}. "
        f"Total de registros brutos: {len(registros_para_inserir)}."
    )

    cursor.execute("""
        INSERT INTO auditoria (
            entidade,
            id_registro,
            operacao,
            descricao,
            data_operacao
        )
        VALUES (?, ?, ?, ?, ?)
    """, (
        "lotes",
        str(id_lote),
        "GERAÇÃO",
        descricao_auditoria,
        agora()
    ))

    # ------------------------------------------------------------
    # 15.13. CONFERÊNCIA ANTES DO COMMIT
    # ------------------------------------------------------------

    quantidade_lote = cursor.execute("""
        SELECT COUNT(*)
        FROM registros_brutos
        WHERE id_lote = ?
    """, (
        id_lote,
    )).fetchone()[0]

    if quantidade_lote != len(registros_para_inserir):
        raise RuntimeError(
            "A conferência do lote encontrou uma quantidade "
            "diferente da esperada."
        )

    # Todas as operações são confirmadas juntas.
    conexao.commit()

    # ------------------------------------------------------------
    # 16. RELATÓRIO FINAL
    # ------------------------------------------------------------

    print("\n" + "=" * 65)
    print("GERAÇÃO CONCLUÍDA COM SUCESSO")
    print("=" * 65)

    print(f"\nVersão do gerador: {VERSAO_GERADOR}")
    print(f"Modo de geração: {MODO_GERACAO}")
    print(f"Lote criado: {id_lote}")

    print("\nANIMAIS")

    print(
        f"Animais originais: {len(animais_originais)}"
    )

    print(
        f"Duplicidades exatas: {len(duplicados_exatos)}"
    )

    print(
        f"Duplicidades parciais: {len(duplicados_parciais)}"
    )

    print(
        f"Total de registros de animais: {len(animais)}"
    )

    print("\nREGISTROS POR CONJUNTO")

    for nome, registros in conjuntos.items():

        print(
            f"{nome}: {len(registros)}"
        )

    print("\nPROBLEMAS INTRODUZIDOS")

    for nome, quantidade in relatorio_problemas.items():

        total = (
            len(animais_originais)
            if nome == "animais"
            else len(conjuntos[nome])
        )

        percentual = (
            quantidade / total * 100
            if total
            else 0
        )

        print(
            f"{nome}: {quantidade} "
            f"({percentual:.2f}%)"
        )

    print("\nTOTAL DE REGISTROS BRUTOS")

    print(len(registros_para_inserir))

    print("\nPRÓXIMO IDENTIFICADOR")

    print(
        f"ANM{proxima_sequencia:08d}"
    )

    print("\nCONFERÊNCIA DO LOTE")

    print(
        f"Registros efetivamente inseridos: {quantidade_inserida}"
    )

    print(
        f"Registros conferidos: {quantidade_lote}"
    )

    print("\n" + "=" * 65)
    print("REGISTROS ANTERIORES PRESERVADOS.")
    print("LOTE REGISTRADO.")
    print("AUDITORIA REGISTRADA.")
    print("BANCO SQLITE ATUALIZADO.")
    print("=" * 65)


except Exception:

    conexao.rollback()

    print("\nERRO DURANTE A GERAÇÃO.")
    print("As operações desta execução foram revertidas.")

    raise

finally:

    conexao.close()


# 17. CONFERÊNCIA FINAL DO BANCO
#
# Esta etapa ocorre depois do fechamento da conexão principal.
# A consulta verifica as tabelas existentes antes de contá-las.
# Não exige que as tabelas tratadas já estejam preenchidas.

conexao = sqlite3.connect(CAMINHO_BANCO)
conexao.row_factory = sqlite3.Row

try:

    tabelas = {
        linha["name"]
        for linha in conexao.execute("""
            SELECT name
            FROM sqlite_master
            WHERE type = 'table'
        """).fetchall()
    }

    tabelas_conferencia = [
        "lotes",
        "registros_brutos",
        "animais",
        "vacinacoes",
        "diagnosticos",
        "maus_tratos",
        "obitos_eutanasias",
        "auditoria",
        "controle_processamento",
        "qualidade_dados",
        "mapeamento_animais"
    ]

    print("\nCONFERÊNCIA GERAL DO BANCO")
    print("=" * 65)

    for tabela in tabelas_conferencia:

        if tabela not in tabelas:
            print(
                f"{tabela}: tabela não encontrada"
            )
            continue

        quantidade = conexao.execute(
            f'SELECT COUNT(*) FROM "{tabela}"'
        ).fetchone()[0]

        print(
            f"{tabela}: {quantidade} registros"
        )

    print("=" * 65)

finally:

    conexao.close()


VERIFICAÇÃO DOS CAMINHOS
PASTA_PROJETO: /content/PROJETO_ZOONOSES
CAMINHO_BANCO: /content/PROJETO_ZOONOSES/banco_zoonoses_v2.db

Pasta do projeto existe?
True

Banco no caminho esperado pelo construtor:
False

Banco no caminho esperado pelo gerador:
False

Arquivos encontrados na pasta do projeto:


FileNotFoundError: Banco SQLite não encontrado. Execute primeiro o construtor do banco.